In [ ]:
import os
import re
import ast
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.corpus import stopwords

# --- [0. 環境與資源初始化] ---
nltk.download('stopwords', quiet=True)

# --- [1. 核心處理函式定義] ---

def load_dictionaries(sc_csv_path, risk_csv_path):
    """載入供應鏈權重字典與風險詞集合"""
    # 處理 SC Dictionary
    df_sc = pd.read_csv(sc_csv_path)
    sc_weights = {}
    for _, row in df_sc.iterrows():
        raw_bigram = row['bigram']
        weight = row['weight']
        try:
            # 支援 tuple 字串 "('supply', 'chain')" 或 "supply_chain"
            if '(' in str(raw_bigram):
                key = ast.literal_eval(raw_bigram)
            elif '_' in str(raw_bigram):
                parts = raw_bigram.lower().split('_')
                key = (parts[0], parts[1])
            else: continue
            sc_weights[key] = weight
        except: continue

    # 處理 Risk Dictionary
    df_risk = pd.read_csv(risk_csv_path)
    risk_set = set(df_risk['Word'].str.lower().str.strip())
    print(list(risk_set)[:10])
    return sc_weights, risk_set


def process_single_txt(text):
    """讀取並清洗文本，回傳標記化的 bigrams"""
    try:
        #with open(file_path, 'r', encoding='utf-8') as f:
        #raw_text = f.read()
        text_clean = re.sub(r'[^a-zA-Z\s]', ' ', text).lower()
        tokens = text_clean.split()
        bigram_sequence = list(zip(tokens, tokens[1:]))
        #將切分好的bigrams加上index，回傳 [(0, (w1, w2)), (1, (w2, w3)), ...] 的格式，以及 bigram 的總數量
        print(f"Processed text into {len(bigram_sequence)} bigrams.")
        return list(enumerate(bigram_sequence)), len(bigram_sequence)
    except Exception as e:
        print(f"讀取檔案 {text} 失敗: {e}")
        return [], 0


def calculate_risk_and_extract_context(indexed_bigrams,
                                       sc_weights,
                                       risk_set,
                                       total_count,
                                       window_size=50):
    """核心邏輯：計算分數 + 擷取 LLM 專用上下文"""
    # 1. 找出風險詞出現的位置(檢查 bigram 中的任一詞是否在 risk_set 中)，是的話就記錄該 bigram 的 index
    risk_indices = [
        idx for idx, (w1, w2) in indexed_bigrams
        if w1 in risk_set or w2 in risk_set
    ]

    # 如果沒有風險詞，直接回傳 0 分和空上下文
    if not risk_indices:
        return 0.0, 0.0, ""  # 沒有風險詞，分數為 0，LLM 上下文為空

    # 2. 計算分數 (使用 window=10 邏輯)
    valid_score_indices = set()
    for r in risk_indices:
        for i in range(max(0, r - 10), min(total_count - 1, r + 10) + 1):
            valid_score_indices.add(i)

    # 若 bigram 的 index 在 valid_score_indices 中，且該 bigram 存在於 sc_weights 中，就將其權重加入加權總和
    weighted_sum = 0.0
    for index, bigram_tuple in indexed_bigrams:
        if index in valid_score_indices and bigram_tuple in sc_weights:
            weighted_sum += sc_weights[bigram_tuple]
            print(
                f"Index {index} with bigram {bigram_tuple} contributes weight {sc_weights[bigram_tuple]}"
            )
        print(f"Current weighted sum: {weighted_sum}")

    # 最終分數 = 加權總和 / bigram 總數量 (如果 bigram 總數量 > 0，否則為 0)
    final_score = weighted_sum / total_count if total_count > 0 else 0.0

    #  3. 擷取 LLM 內容：從第一個風險詞前 window_size 到最後一個風險詞後 window_size
    # first_risk = min(risk_indices)
    # last_risk = max(risk_indices)

    # start = max(0, first_risk - window_size)
    # end = min(total_count - 1, last_risk + window_size)

    matched_sc_indices = [
        index for index, bigram_tuple in indexed_bigrams
        if index in valid_score_indices and bigram_tuple in sc_weights
    ]

    if not matched_sc_indices:
        return 0.0, 0.0, ""

    start = max(0, min(matched_sc_indices) - window_size)
    end = min(total_count - 1, max(matched_sc_indices) + window_size)

    # 重新組合單字（從 start 到 end 的連續片段）
    seg = [indexed_bigrams[i][1][0] for i in range(start, end + 1)]
    seg.append(indexed_bigrams[end][1][1])
    llm_context = " ".join(seg)
    return final_score, weighted_sum, llm_context



def find_txt_file(base_path, filename):
    """
    在 base_path 及其子資料夾中搜尋指定的 txt 檔案
    返回: (完整路徑, 事件類型) 或 (None, None)
    """
    # 移除可能的副檔名，確保搜尋 .txt
    base_name = filename.replace('.txt', '')

    # 遍歷所有子資料夾
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file == f"{base_name}.txt" or file == filename:
                event_type = os.path.basename(root)
                full_path = os.path.join(root, file)
                return full_path, event_type

    return None, None


In [3]:
# 設定路徑
sc_path = r'C:\Users\user\Desktop\ravenpack\supplychain risk\full_bigrams_cleaned.csv'
risk_path = r'C:\Users\user\Desktop\ravenpack\supplychain risk\hassan\risk.csv'
txt_base_path = r'C:\Users\user\Desktop\ravenpack\DATA\target_data\2024'
input_master_csv = r'C:\Users\user\Desktop\ravenpack\DATA\target_data\整年csv存檔\2024\All_Years_Grand_Total_Riskwords_2024_all_new.csv'  # 總檔CSV
output_csv = r'C:\Users\user\Desktop\ravenpack\supplychain risk\output\llm_ready_整年\2024\llm_ready_data_2024_100words_v3(TS).csv'

# 執行字典載入與處理
sc_weights, risk_set = load_dictionaries(sc_path, risk_path)

['risky', 'doubtful', 'danger', 'torn', 'vacillation', 'debatable', 'doubtfulness', 'ambivalence', 'risking', 'peril']


In [4]:
# 測試文本
test = 'The company faces high uncertainty regarding future production cost variations. Also, unexpected logistics bottlenecks create major risk for our current transportation cost planning.'

In [5]:
sc_weights

{('supply', 'chain'): 761.6278817,
 ('the', 'supply'): 281.1494923,
 ('a', 'supply'): 146.2325533,
 ('the', 'retailer'): 133.1760753,
 ('of', 'demand'): 104.8870397,
 ('the', 'manufacturer'): 104.8870397,
 ('lead', 'time'): 98.79401665,
 ('demand', 'is'): 93.13620953,
 ('of', 'product'): 79.20929969,
 ('the', 'demand'): 74.42192444,
 ('the', 'supplier'): 71.81062884,
 ('transportation', 'cost'): 64.41195799,
 ('of', 'supply'): 56.57807121,
 ('transportation', 'costs'): 56.57807121,
 ('an', 'order'): 55.70763935,
 ('expected', 'profit'): 53.53155968,
 ('demand', 'and'): 52.66112782,
 ('third', 'party'): 52.66112782,
 ('supply', 'chains'): 52.66112782,
 ('fixed', 'cost'): 46.56810477,
 ('the', 'season'): 45.26245697,
 ('the', 'quantity'): 44.82724104,
 ('demand', 'in'): 40.91029764,
 ('and', 'demand'): 39.60464985,
 ('of', 'transportation'): 38.73421798,
 ('revenue', 'management'): 38.73421798,
 ('chain', 'management'): 38.29900205,
 ('response', 'time'): 37.42857019,
 ('demand', 'uncert

In [6]:
indexed_bg, total_cnt = process_single_txt(test)
if total_cnt > 0:
        score, w_sum, context = calculate_risk_and_extract_context(
            indexed_bg, sc_weights, risk_set, total_cnt)

Processed text into 22 bigrams.
Current weighted sum: 0.0
Current weighted sum: 0.0
Current weighted sum: 0.0
Current weighted sum: 0.0
Current weighted sum: 0.0
Current weighted sum: 0.0
Index 6 with bigram ('future', 'production') contributes weight 0.435215932
Current weighted sum: 0.435215932
Index 7 with bigram ('production', 'cost') contributes weight 18.27906916
Current weighted sum: 18.714285091999997
Current weighted sum: 18.714285091999997
Current weighted sum: 18.714285091999997
Current weighted sum: 18.714285091999997
Current weighted sum: 18.714285091999997
Current weighted sum: 18.714285091999997
Current weighted sum: 18.714285091999997
Current weighted sum: 18.714285091999997
Index 15 with bigram ('major', 'risk') contributes weight 0.435215932
Current weighted sum: 19.149501023999996
Index 16 with bigram ('risk', 'for') contributes weight 1.305647797
Current weighted sum: 20.455148820999995
Current weighted sum: 20.455148820999995
Current weighted sum: 20.45514882099999

In [11]:
indexed_bg

[(0, ('the', 'company')),
 (1, ('company', 'faces')),
 (2, ('faces', 'high')),
 (3, ('high', 'uncertainty')),
 (4, ('uncertainty', 'regarding')),
 (5, ('regarding', 'future')),
 (6, ('future', 'production')),
 (7, ('production', 'cost')),
 (8, ('cost', 'variations')),
 (9, ('variations', 'also')),
 (10, ('also', 'unexpected')),
 (11, ('unexpected', 'logistics')),
 (12, ('logistics', 'bottlenecks')),
 (13, ('bottlenecks', 'create')),
 (14, ('create', 'major')),
 (15, ('major', 'risk')),
 (16, ('risk', 'for')),
 (17, ('for', 'our')),
 (18, ('our', 'current')),
 (19, ('current', 'transportation')),
 (20, ('transportation', 'cost')),
 (21, ('cost', 'planning'))]

In [7]:
#算最後分數
print(score)
print(w_sum)
print(context)

3.8575957641363634
84.867106811
the company faces high uncertainty regarding future production cost variations also unexpected logistics bottlenecks create major risk for our current transportation cost planning
